# BranchCache — free-tier GPU serving benchmark

Runs the naive / vLLM / SGLang serving comparison from `eval/run_serving_benchmark.py` against a real model. No paid services anywhere — this notebook only needs the free GPU tier, no billing info required.

Default model: `Qwen/Qwen2.5-Coder-3B-Instruct` (set `BRANCHCACHE_MODEL` to override, e.g. drop to `Qwen/Qwen2.5-Coder-1.5B-Instruct` if you OOM at high N).


**Kaggle-specific:** Notebook settings → Accelerator → GPU T4 x2 (only one GPU is used here, but T4 x2 is what's offered) and Internet → On (needed for `pip install` and the git clone). Free quota is a 9-hour session limit and 30 GPU-hours/week — if the sweep doesn't finish in one session, just start a fresh session and re-run the notebook; completed cells in the CSV are skipped.

## 1. Clone the repo and install

In [1]:
import os

REPO_URL = "https://github.com/OmkarKashyap/BranchCache.git"
if not os.path.exists("BranchCache"):
    !git clone $REPO_URL
%cd BranchCache
!pip install -q -e ".[dev,gpu]"


Cloning into 'BranchCache'...
remote: Enumerating objects: 38, done.
remote: Counting objects: 100% (38/38), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 38 (delta 2), reused 35 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (38/38), 15.90 KiB | 3.97 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/kaggle/working/BranchCache
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 790.6/790.

In [ ]:
!nvidia-smi

Sanity check — the CPU-only unit tests should still pass here, same as on your laptop:

In [ ]:
!pytest tests/ -q

## 2. Helpers

One server at a time on a single free-tier GPU keeps memory pressure predictable — we bring a server up, run that strategy's cell of the sweep, tear it down, then move to the next. `run_serving_benchmark.py` checkpoints to CSV as it goes, so each of these three runs just appends its own rows; nothing gets re-run if a cell is re-executed.

In [ ]:
import os
import subprocess
import time

import requests


def start_server(cmd, log_path):
    log = open(log_path, "w")
    return subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)

def wait_for_server(port, log_path, timeout_s=600):
    url = f"http://localhost:{port}/health"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            if requests.get(url, timeout=2).status_code == 200:
                print(f"server on port {port} is up")
                return
        except Exception:
            pass
        time.sleep(5)
    raise RuntimeError(f"server on port {port} never came up, check {log_path}")

def stop_server(proc):
    proc.terminate()
    try:
        proc.wait(timeout=15)
    except subprocess.TimeoutExpired:
        proc.kill()


## 3. Naive baseline (prefix caching disabled)

Same vLLM server as the caching run below, just started with `--no-enable-prefix-caching` — this isolates caching as the one thing that changes between the two runs.

In [ ]:
%env BRANCHCACHE_MODEL=Qwen/Qwen2.5-Coder-3B-Instruct
%env BRANCHCACHE_NAIVE_URL=http://localhost:8000/v1

naive_proc = start_server(
    ['vllm', 'serve', os.environ['BRANCHCACHE_MODEL'], '--port', '8000',
     '--no-enable-prefix-caching', '--gpu-memory-utilization', '0.85',
     '--max-model-len', '8192'],
    'naive.log',
)
wait_for_server(8000, 'naive.log')


In [ ]:
!python -m eval.run_serving_benchmark --strategies naive --ns 1 2 4 8 --trials 3

In [ ]:
stop_server(naive_proc)

## 4. vLLM automatic prefix caching

In [ ]:
%env BRANCHCACHE_VLLM_URL=http://localhost:8001/v1

vllm_proc = start_server(
    ['vllm', 'serve', os.environ['BRANCHCACHE_MODEL'], '--port', '8001',
     '--enable-prefix-caching', '--gpu-memory-utilization', '0.85',
     '--max-model-len', '8192'],
    'vllm.log',
)
wait_for_server(8001, 'vllm.log')


In [ ]:
!python -m eval.run_serving_benchmark --strategies vllm --ns 1 2 4 8 --trials 3

In [ ]:
stop_server(vllm_proc)

## 5. SGLang RadixAttention

RadixAttention is on by default in SGLang, no extra flag needed.

In [ ]:
%env BRANCHCACHE_SGLANG_URL=http://localhost:8002/v1

sglang_proc = start_server(
    ['python', '-m', 'sglang.launch_server', '--model-path', os.environ['BRANCHCACHE_MODEL'],
     '--port', '8002'],
    'sglang.log',
)
wait_for_server(8002, 'sglang.log')


In [ ]:
!python -m eval.run_serving_benchmark --strategies sglang --ns 1 2 4 8 --trials 3

In [ ]:
stop_server(sglang_proc)

## 6. Done

`results/raw/serving_benchmark.csv` now has rows for all three strategies. Pull it back down (or commit it from here if this environment has your git credentials) — that CSV is what `scripts/plot_results.py` reads in a later phase.

If the session dies partway through, just re-run this notebook top to bottom — completed (problem, N, strategy, trial) cells are skipped automatically.